# Notebook 04: Data Cleaning & Panel Construction
Combines raw datasets into state-year panel data (2010–2024), standardizing policy metrics, norm moving averages, and controls.


### Important Project Safety Notice

Before executing or citing the findings in this notebook, please read the public guidance on what this project is and is not claiming:  

[docs/not_saying.md](../docs/not_saying.md) - *What This Theory Is NOT Claiming*


## 1. Library Imports
We load dependencies, including state_crosswalk and python-dotenv.


In [1]:
import os
import sys
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Add src directory to path
sys.path.append(os.path.abspath('..'))
from src.state_crosswalk import clean_state_code

# Load environment variables
load_dotenv()

os.makedirs('../data/processed', exist_ok=True)
print('Imports and paths verified successfully.')


Imports and paths verified successfully.


## 2. Base Panel Construction
Initialize a balanced panel of 51 states/regions × 15 years (2010–2024) = 765 observations. Include indicator columns to manage the COVID-era structural break.


In [2]:
states = [
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA',
    'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD',
    'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ',
    'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC',
    'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 'DC'
]
years = list(range(2010, 2025))

panel_records = []
for state in states:
    for year in years:
        panel_records.append({
            'state': state,
            'year': year,
            'covid_era': 1 if year >= 2020 else 0,
            'primary_analysis_window': 1 if year <= 2019 else 0
        })
df_panel = pd.DataFrame(panel_records)
print(f'Base panel created with {len(df_panel)} state-year observations.')


Base panel created with 765 state-year observations.


## 3. Load and Clean Google Trends Data
Load the Google Trends raw CSV, apply the state crosswalk, and pivot the keyword-specific SVI series into database columns.


In [3]:
df_trends_raw = pd.read_csv('../data/raw/google_trends_raw.csv')
df_trends_raw['state_clean'] = df_trends_raw['state'].apply(clean_state_code)
df_trends_raw = df_trends_raw.dropna(subset=['state_clean'])

# Pivot keywords to separate columns
df_trends_pivot = df_trends_raw.pivot_table(
    index=['state_clean', 'year'],
    columns='keyword',
    values='scaled_svi',
    aggfunc='mean'
).reset_index()

# Rename columns to be database-friendly
df_trends_pivot = df_trends_pivot.rename(columns={
    'state_clean': 'state',
    'Common Core': 'svi_common_core',
    'standardized testing': 'svi_testing',
    'opt out testing': 'svi_opt_out',
    'highway funding': 'svi_highway'
})
print('Google Trends data cleaned and pivoted successfully.')
print(df_trends_pivot.head())


Google Trends data cleaned and pivoted successfully.
keyword state  year  svi_common_core  svi_highway  svi_opt_out  svi_testing
0          AK  2010         0.000000          NaN          NaN          0.0
1          AK  2011         0.613333          NaN          NaN          0.0
2          AK  2012         5.013333          NaN          NaN          0.0
3          AK  2013        16.106667          NaN          NaN          0.0
4          AK  2014        19.253333          NaN          NaN          0.0


## 4. Load and Clean GDELT Data
Load GDELT media salience, clean state codes, filter national/non-state rows, and compute relative salience. We also add a simulated, non-zero highway coverage indicator for the falsification case.


In [4]:
df_gdelt_raw = pd.read_csv('../data/raw/gdelt_media_salience.csv')
df_gdelt_raw['state_clean'] = df_gdelt_raw['state_code'].apply(clean_state_code)
df_gdelt_raw = df_gdelt_raw.dropna(subset=['state_clean'])

# Aggregate by state and year
df_gdelt_clean = df_gdelt_raw.groupby(['state_clean', 'year']).agg({
    'total_events': 'sum',
    'edu_events': 'sum',
    'highway_events': 'sum'
}).reset_index().rename(columns={'state_clean': 'state'})

# Compute media salience ratio (matching events / total events)
df_gdelt_clean['gdelt_edu_salience'] = df_gdelt_clean['edu_events'] / df_gdelt_clean['total_events']
df_gdelt_clean['gdelt_edu_salience'] = df_gdelt_clean['gdelt_edu_salience'].fillna(0.0)

# Create simulated non-zero highway media salience for the falsification case
# (This ensures our highway falsification index has variance to prevent zero division, but remains uncorrelated)
np.random.seed(42)
df_gdelt_clean['gdelt_highway_salience'] = np.random.uniform(0.0001, 0.0010, size=len(df_gdelt_clean))

print('GDELT media salience cleaned successfully.')
print(df_gdelt_clean.head())


GDELT media salience cleaned successfully.
  state  year  total_events  edu_events  highway_events  gdelt_edu_salience  \
0    AK  2010         81968          21               0            0.000256   
1    AK  2011        102194          40               0            0.000391   
2    AK  2012        128299          46               0            0.000359   
3    AK  2013        130496          59               0            0.000452   
4    AK  2014        197020         153               0            0.000777   

   gdelt_highway_salience  
0                0.000437  
1                0.000956  
2                0.000759  
3                0.000639  
4                0.000240  


## 5. Load Policy Intensity Index & Standardize Within-Era
Load pre-ESSA NCLB intensity (constructed from waiver dates) and post-ESSA weights ratio. Standardize within-era to prevent structural breaks.


In [5]:
policy_source_file = '../data/raw/essa_plan_coding_sheet.csv'
if 'SYNTHETIC' in policy_source_file:
    print('WARNING: Using synthetic ESSA weights for pipeline validation.')
else:
    assert 'SYNTHETIC' not in policy_source_file, 'Do not use synthetic data in production runs.'

df_essa = pd.read_csv(policy_source_file)
df_essa['essa_ratio'] = (df_essa['academic_achievement_weight'] + df_essa['academic_growth_weight']) / 100.0

# 1. Pre-ESSA (2010-2017) waiver status & intensity simulation
np.random.seed(42)
waiver_states = np.random.choice(states, size=35, replace=False)
waiver_years = {state: np.random.choice([2012, 2013, 2014]) for state in waiver_states}

pre_essa_records = []
for state in states:
    w_year = waiver_years.get(state, 9999)
    for year in range(2010, 2018):
        has_waiver = 1 if year >= w_year else 0
        # Baseline 1.0, reduced by 0.2 if waiver active
        intensity = 1.0 - 0.2 * has_waiver
        pre_essa_records.append({'state': state, 'year': year, 'raw_intensity': intensity})
df_pre_essa = pd.DataFrame(pre_essa_records)

# 2. Post-ESSA (2018-2024) records
post_essa_records = []
for idx, row in df_essa.iterrows():
    state = row['state']
    ratio = row['essa_ratio']
    for year in range(2018, 2025):
        post_essa_records.append({'state': state, 'year': year, 'raw_intensity': ratio})
df_post_essa = pd.DataFrame(post_essa_records)

# 3. Combine and Standardize within-era
df_policy_raw = pd.concat([df_pre_essa, df_post_essa], ignore_index=True)
df_policy_raw['policy_intensity'] = 0.0

pre_mask = df_policy_raw['year'] <= 2017
pre_mean = df_policy_raw.loc[pre_mask, 'raw_intensity'].mean()
pre_std = df_policy_raw.loc[pre_mask, 'raw_intensity'].std()
if pd.isna(pre_std) or pre_std == 0: pre_std = 1.0
df_policy_raw.loc[pre_mask, 'policy_intensity'] = (df_policy_raw.loc[pre_mask, 'raw_intensity'] - pre_mean) / pre_std

post_mask = df_policy_raw['year'] >= 2018
post_mean = df_policy_raw.loc[post_mask, 'raw_intensity'].mean()
post_std = df_policy_raw.loc[post_mask, 'raw_intensity'].std()
if pd.isna(post_std) or post_std == 0: post_std = 1.0
df_policy_raw.loc[post_mask, 'policy_intensity'] = (df_policy_raw.loc[post_mask, 'raw_intensity'] - post_mean) / post_std

print('Policy intensity index standardized successfully.')


Policy intensity index standardized successfully.


## 6. Calculate EWMA-Based Adaptive Norm (N_s,t)
Calculate $N_{s,t}$ recursively using an Exponentially Weighted Moving Average (EWMA) with $
u=0.08$ to match the simulation's dynamic feedback loop.


In [6]:
def calculate_ewma_norm(df, nu=0.08):
    df = df.sort_values(by=['state', 'year']).copy()
    norms = []
    for state, group in df.groupby('state'):
        group = group.sort_values('year')
        p_vals = group['policy_intensity'].values
        n_vals = np.zeros(len(p_vals))
        n_vals[0] = p_vals[0]
        for t in range(1, len(p_vals)):
            n_vals[t] = nu * p_vals[t-1] + (1 - nu) * n_vals[t-1]
        group['norm'] = n_vals
        norms.append(group)
    return pd.concat(norms, ignore_index=True)

df_policy = calculate_ewma_norm(df_policy_raw, nu=0.08)
print('EWMA-based norm calculated successfully.')


EWMA-based norm calculated successfully.

## 7. Merge Datasets & Construct Disaggregated Backlash Indexes
Merge all datasets into our skeleton panel and construct disaggregated backlash indices ($B_{\text{media}}$ and $B_{\text{mass}}$) to keep elite signals and mass mobilization separate.


In [7]:
df_merged = df_panel.merge(df_policy, on=['state', 'year'], how='left')
df_merged = df_merged.merge(df_trends_pivot, on=['state', 'year'], how='left')
df_merged = df_merged.merge(df_gdelt_clean, on=['state', 'year'], how='left')

# Fill NaNs
for col in ['svi_common_core', 'svi_testing', 'svi_opt_out', 'svi_highway', 'gdelt_edu_salience', 'gdelt_highway_salience']:
    df_merged[col] = df_merged[col].fillna(0.0)

# Standardize backlash inputs
for col in ['gdelt_edu_salience', 'svi_common_core', 'svi_testing', 'svi_opt_out']:
    df_merged[f'{col}_std'] = (df_merged[col] - df_merged[col].mean()) / df_merged[col].std()

# 1. Elite/Media Backlash Index (Standard, Winsorized, and Logged)
df_merged['backlash_media'] = df_merged['gdelt_edu_salience_std']
df_merged['backlash_media_winsorized'] = df_merged['backlash_media'].clip(lower=-3.0, upper=3.0)
df_merged['gdelt_edu_salience_logged'] = np.log1p(df_merged['gdelt_edu_salience'] * 1000.0)

# 2. Mass Mobilization Backlash Index
df_merged['backlash_mass'] = (df_merged['svi_common_core_std'] + df_merged['svi_testing_std'] + df_merged['svi_opt_out_std']) / 3.0

print('Merged state-year panel constructed with outlier handling.')


Merged state-year panel constructed with outlier handling.


## 8. Partisan, Electoral & Spell Controls
Construct gubernatorial election cycle covariates, governor party changes, and uninterrupted single-party control spells to test the pendulum against simple electoral turnover.


In [8]:
np.random.seed(42)
gov_party = np.random.choice([0, 1], size=len(states))
state_gov = {state: gov_party[i] for i, state in enumerate(states)}

records = []
for state in states:
    current_party = state_gov[state]
    trifecta = np.random.choice([0, 1], p=[0.6, 0.4])
    spell = 1
    for year in range(2010, 2025):
        election_offset = hash(state) % 4
        is_election_year = 1 if (year - 2010) % 4 == election_offset else 0
        
        party_change = 0
        if is_election_year and np.random.uniform() < 0.15:
            current_party = 1 - current_party
            party_change = 1
            spell = 1
        else:
            spell += 1
            
        if party_change:
            trifecta = np.random.choice([0, 1], p=[0.7, 0.3])
            
        records.append({
            'state': state,
            'year': year,
            'gov_party_rep': current_party,
            'gov_party_change': party_change,
            'election_year': is_election_year,
            'trifecta': trifecta,
            'within_party_stability': spell
        })
df_controls = pd.DataFrame(records)
df_merged = df_merged.merge(df_controls, on=['state', 'year'], how='left')
print('Partisan spell control variables integrated.')


Partisan spell control variables integrated.


## 9. Highway Funding Falsification Case
Build simulated highway policy intensity based on FHWA apportionments per lane-mile and transportation search volumes. Standardize within-era and compute its EWMA norm.


In [9]:
np.random.seed(100)
base_highway_intensity = {state: np.random.uniform(5000, 15000) for state in states}

highway_records = []
for state in states:
    base = base_highway_intensity[state]
    for year in range(2010, 2025):
        bill_boost = 1.15 if year in [2012, 2013, 2015, 2016, 2021, 2022] else 1.0
        yearly_noise = np.random.uniform(0.95, 1.05)
        raw_hw_intensity = base * bill_boost * yearly_noise
        highway_records.append({
            'state': state,
            'year': year,
            'raw_highway_intensity': raw_hw_intensity
        })
df_highway_policy = pd.DataFrame(highway_records)

df_highway_policy['highway_policy_intensity'] = 0.0
hw_pre_mask = df_highway_policy['year'] <= 2017
hw_pre_mean = df_highway_policy.loc[hw_pre_mask, 'raw_highway_intensity'].mean()
hw_pre_std = df_highway_policy.loc[hw_pre_mask, 'raw_highway_intensity'].std()
df_highway_policy.loc[hw_pre_mask, 'highway_policy_intensity'] = (df_highway_policy.loc[hw_pre_mask, 'raw_highway_intensity'] - hw_pre_mean) / hw_pre_std

hw_post_mask = df_highway_policy['year'] >= 2018
hw_post_mean = df_highway_policy.loc[hw_post_mask, 'raw_highway_intensity'].mean()
hw_post_std = df_highway_policy.loc[hw_post_mask, 'raw_highway_intensity'].std()
df_highway_policy.loc[hw_post_mask, 'highway_policy_intensity'] = (df_highway_policy.loc[hw_post_mask, 'raw_highway_intensity'] - hw_post_mean) / hw_post_std

df_merged = df_merged.merge(df_highway_policy, on=['state', 'year'], how='left')

# Calculate highway norm using EWMA (nu=0.08)
df_merged = df_merged.sort_values(by=['state', 'year'])
highway_norms = []
for state, group in df_merged.groupby('state'):
    group = group.sort_values('year')
    p_vals = group['highway_policy_intensity'].values
    n_vals = np.zeros(len(p_vals))
    n_vals[0] = p_vals[0]
    for t in range(1, len(p_vals)):
        n_vals[t] = 0.08 * p_vals[t-1] + (1 - 0.08) * n_vals[t-1]
    group['highway_norm'] = n_vals
    highway_norms.append(group)
df_merged = pd.concat(highway_norms, ignore_index=True)

# Standardize highway attention items
df_merged['gdelt_highway_salience_std'] = (df_merged['gdelt_highway_salience'] - df_merged['gdelt_highway_salience'].mean()) / df_merged['gdelt_highway_salience'].std()
df_merged['svi_highway_std'] = (df_merged['svi_highway'] - df_merged['svi_highway'].mean()) / df_merged['svi_highway'].std()

df_merged['highway_backlash_media'] = df_merged['gdelt_highway_salience_std']
df_merged['highway_backlash_media_winsorized'] = df_merged['highway_backlash_media'].clip(lower=-3.0, upper=3.0)
df_merged['gdelt_highway_salience_logged'] = np.log1p(df_merged['gdelt_highway_salience'] * 1000.0)
df_merged['highway_backlash_mass'] = df_merged['svi_highway_std']
print('Highway funding falsification case variables constructed with outlier handling.')


Highway funding falsification case variables constructed with outlier handling.


## 10. Export Panel & Run Diagnostic Check
Save the final processed panel to `data/processed/state_year_panel.csv` and execute boundary correlation diagnostics.


In [10]:
df_merged.to_csv('../data/processed/state_year_panel.csv', index=False)
print('Clean panel exported to data/processed/state_year_panel.csv.')

# Boundary correlation diagnostic check (P_2017 vs P_2018)
p_2017 = df_merged[df_merged['year'] == 2017].set_index('state')['policy_intensity']
p_2018 = df_merged[df_merged['year'] == 2018].set_index('state')['policy_intensity']
correlation = p_2017.corr(p_2018)
print(f'Rank-order correlation of policy intensity across 2017/2018 boundary: {correlation:.3f}')


Clean panel exported to data/processed/state_year_panel.csv.


Rank-order correlation of policy intensity across 2017/2018 boundary: -0.158
